# Spark SQL Transformation


## DSL approach Vs declarative (SQL syntax)
## performance wise no difference - DSL/SQL/DSL+SQL - internally its going run in RDDway
## 
## generally we will mix both DSL + SQL 
## mostly reading the data go with DSL approach
## and the transformation part people will go with SQL approach

read customer data - input custid, fname,lname,age, profession

what i want - cust_id, fullname,age,is_vote,upper(profession),data_dt,createdby

In [0]:
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/custs_header",header=True,inferSchema=True)
cust_df.show(5)

In [0]:

#create fullname
#mapping

"""
1. custid- direct mapping
2. fullname - concat(fname,lname)
3.age-direct mapping
4. is_vote= derive from age column
5.profession - upper(prof)
6.data_dt= currentDate()
7.createdBy=hard code to anu

"""
cust_df.show(5)
cust_df.printSchema()



## Create temporary view of DF with SQL

using CreateorReplaceTempView

In [0]:
#create temporary view using CreateorReplaceTempView
cust_df.createOrReplaceTempView("View_cust")

spark.sql("select * from View_cust").show(5)   #output of spark.sql is a dataframe

In [0]:
sql_string="select custid,concat(fname,' ',lname) as fullname,age, case when age>=18 then 'Yes' else 'No' end as is_Vote,upper(profession)as Profession,current_date() as date_dt,'anu' as createdBy from View_cust"

spark.sql(sql_string).show(5)

In [0]:

spark.sql("select Profession,count(1) as tot_count from View_cust group by Profession").orderBy("Profession",ascending=False).show(5)
spark.sql("select Profession,count(1) as tot_count from View_cust group by Profession").show(5)

spark.sql("select Profession,count(1) as tot_count from View_cust group by Profession").orderBy("Profession",ascending=True).show(5)

spark.sql("select age,Profession,count(1) as tot_count from View_cust group by age,Profession").orderBy("age",ascending=False).show(5)

In [0]:

# shows tables/ views in default database
spark.sql("show tables").show()

# shows tables/ views in particular database
spark.sql("show tables in izwd37dev.wd37db").show()

### Create spark session 

In [0]:
from pyspark.sql import SparkSession

spark1=SparkSession.builder.getOrCreate()
print(spark1)
print(spark)

## Create view using pure SQL

In [0]:
%sql

-- create view using SQL syntax

create or replace temporary view cust_view
using csv
options (
    header="true",
    inferSchema="True",
    path="/Volumes/izwd37dev/wd37db/rawdatta/csv/custs_header"
);
select * from cust_view limit 10;






In [0]:
sql_str="""
create or replace temporary view cust_view1
using csv
options (
    header="true",
    inferSchema="True",
    path="/Volumes/izwd37dev/wd37db/rawdatta/csv/custs_header"
);"""
spark.sql(sql_str)
spark.sql("select * from cust_view1")


In [0]:
spark.sql("select * from cust_view1").show(5)
spark.sql("select * from cust_view1 limit 10").show()

spark.sql() = sql()

In [0]:
sql("select * from cust_view limit 10").show()

In [0]:
print(sql)
print(spark.sql)

In [0]:

#taking the default database
display(sql("show tables "))

#specifying specific database
display(sql("show tables in izwd37dev.wd37db "))

## Create Schema in SQL

### Read CSV

In [0]:
%sql
-- create schema in SQL

-- custid|   fname|     lname|age|          profession|
create or replace temporary view cust_view
(
    cust_id int,
    fname string,
    lname string,
    age int,
    Prof string
)
using csv
options (
    header="true",
    inferSchema="True",
    path="/Volumes/izwd37dev/wd37db/rawdatta/csv/custs_header"
);

select * from cust_view limit 10;

### Read CSV with mode Permissive

In [0]:
%sql
create or replace temporary view cust_view_perm
(
    cust_id int,
    fname string,
    lname string,
    age int,
    Prof string,
    _errors string
)
using csv
options (
    header="true",
    inferSchema="True",
    path="/Volumes/izwd37dev/wd37db/rawdatta/csv/custsmodified.csv",
    mode="PERMISSIVE",
    columnNameOfCorruptRecord="_errors"
);
select * from cust_view_perm limit 10;


%md
In orrder to transform the data using sql query ,

1. we need to create dataframe from the source (csv/parquet/json..) using DSL is the prefred  (spark.read)

2. df.createOrReplaceTemView using this we can create temp view 

3. write our transform sql query and call using spark.sql or sql()  

### Read CSV with DropMalFormed

In [0]:
%sql
create or replace temporary view cust_view_dropMal
(
    cust_id int,
    fname string,
    lname string,
    age int,
    Prof string,
    _errors string
)
using csv
options (
    header="true",
    inferSchema="True",
    path="/Volumes/izwd37dev/wd37db/rawdatta/csv/custsmodified.csv",
    mode="dropMalformed",
    columnNameOfCorruptRecord="_errors"
);
select * from cust_view_dropMal limit 10;

### Read JSON

In [0]:
%sql
-- read json data using SQL 
-- /Volumes/izwd37dev/wd37db/rawdatta/json/emp.json

create or replace temporary view emp_view
using json
options (
    multiline="true",
    path="/Volumes/izwd37dev/wd37db/rawdatta/json/emp.json"
    
);
select * from emp_view limit 10


In [0]:
df=spark.sql("select * from emp_view limit 10")
df.printSchema()
df.show()

In [0]:
%sql
show tables in izwd37dev.wd37db

###  Read a permanent table using sql syntax

In [0]:
%sql
select * from izwd37dev.wd37db.cust_info_table

In [0]:
table_df=spark.read.table("izwd37dev.wd37db.cust_info_table")
table_df.show()

# Show Columns list and table info

## Describe, formatted, extended, detail

In [0]:
%sql 
describe izwd37dev.wd37db.cust_info_table;   --similar to print(desc_df.describe()) -- provides column details in table

describe formatted izwd37dev.wd37db.cust_info_table; -- provides some detailed table information
describe extended izwd37dev.wd37db.cust_info_table; -- provides both columns list and detailed table information

describe detail izwd37dev.wd37db.cust_info_table; -- provides more details about db like partitioncolumns, bytes etc.


In [0]:
#the below are similar to the above SQL syntax

desc_df=spark.sql("select * from izwd37dev.wd37db.cust_info_table")
print(desc_df.columns)
print(desc_df.schema)
print(desc_df.describe())
desc_df.summary().show()

In [0]:
%sql
select * from izwd37dev.wd37db.cust_info_table limit 5;
select count(cust_id),max(cust_id),min(cust_id) from izwd37dev.wd37db.cust_info_table;



# union , union all , unionByName(specific to DSL)

##union - combine same type of data / same strucure from two diff table , remove duplicate in SQL
### union (dsl) -combine same type of data / same strucure from two diff table , all data  (similar to union all) - not remove duplicates

# union all - combine same type of data / same strucure from two diff table , all data 

# unionByname - only available in dsl 



### Union

In [0]:
#combine Data
#union , union all , unionByName(specific to DSL)

student_df1=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part1.csv",header=True,inferSchema=True)
student_df1.createOrReplaceTempView("stud_view1")


student_df2=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part2.csv",header=True,inferSchema=True)
student_df2.createOrReplaceTempView("stud_view2")

student_df3=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part3.csv",header=True,inferSchema=True)
student_df3.createOrReplaceTempView("stud_view3")

student_df4=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part4.csv",header=True,inferSchema=True)
student_df4.createOrReplaceTempView("stud_view4")

spark.sql("select * from stud_view1").show()
spark.sql("select * from stud_view2").show()
spark.sql("select * from stud_view3").show()
spark.sql("select * from stud_view4").show()

#union
union_query="""
select * from stud_view3
union
select * from stud_view4
"""

spark.sql(union_query).show()    #removes duplicates since its a SQL command

union_df=student_df3.union(student_df4)
union_df.show()                    # it will not removes duplicates since its a DSL command
 






In [0]:
%sql

select * from stud_view3 union select * from stud_view4

### Union All

union all - combine same type of data / same strucure from two diff table , all data (will not remove duplicates)

In [0]:
#union all

# no union all in SQL
# Union all available only in Spark SQL
unionall_query="""
select * from stud_view3
union all
select * from stud_view4
"""

spark.sql(unionall_query).show()   #will not remove duplicates

unionall_df=student_df3.unionAll(student_df4)     #will not remove duplicates
unionall_df.show()


### Union By Name

In [0]:
#union by name is not an option in SQL
#it is available only in DSL
unionByName_df=student_df2.unionByName(student_df4,allowMissingColumns=True)
unionByName_df.show()


# similar to union by name in SQL without function - just implemented the logic alone

unionByName_query="""
select sid,sname,year,null as city from stud_view2
union
select sid,sname,null as year,city from stud_view4
"""
sql(unionByName_query).show()

# 2. validation , cleansing , scrubbing  



### validation (na.drop) - handle missing value, handling null 

In [0]:
data=[
    (100,"crish",25),
    (101,"bala",None),
    (102,None,None),
    (None,None,None),
    (103,"raja",None),
    (104,None,25),
]
data_df=spark.createDataFrame(data,["id","fname","age"])
data_df.show()

#create view to use in SQLcommand
data_df.createOrReplaceTempView("data_view")
spark.sql("select * from data_view").show()


#handling null values
#na.drop(how,subset) - any is the default parameter for how
data_df.na.drop().show()
data_df.na.drop(how="any").show()  # remove if any column have NULL value


#how="any"
# sql 
# if all column has value retrun that record 

spark.sql("select * from data_view where id is not null and fname is not null and age is not null").show()



#how="all"
data_df.na.drop(how="all").show()    
# remove if all column have null
#how="all"
spark.sql("select * from data_view where id is not null or fname is not null or age is not null").show()



#subset=[]
#remove the records of specified column is null
data_df.na.drop(subset=["id","fname"]).show()
spark.sql("select * from data_view where id is not null and fname is not null").show()











## Scrubbing - Filling(na.fill)

nvl 

coalesce

nvl2(null,if null , if not null)

### nvl (id,0)
NVL(expression1, expression2)

If expression1 is NOT NULL, it returns expression1.

If expression1 is NULL, it returns expression2.


In [0]:
data_df.na.fill(0).show()       #for integer columns
data_df.na.fill("unknown").show()            #for string columns
data_df.na.fill("unknown",subset=["fname"]).show()          #for string specific columns


#implement the same in SQL syntax
data_df.createOrReplaceTempView("data_view")
spark.sql("select * from data_view").show()
spark.sql("select nvl(id,0)as id,nvl(fname,'unknown')as fname,nvl(age,0)as age from data_view").show()



### coalesce - similar to nvl function

NVL() and COALESCE() are both used to replace NULL values.

whereas NVL() number of arguments is 2, so if expression 1 is null it takes expression 2

in Coalesce() number of arguments is more, so if expression1,2 is null it takes first non null expression

In [0]:
spark.sql("select * from data_view").show()
spark.sql("select coalesce(id,0)as id,coalesce(fname,'unknown')as fname,coalesce(age,0)as age from data_view").show();

### nvl2 - similar like if elseif

nvl2(expression1,expression2,expression3)

if expression 1 is not null use expression 2

if expression 1 is null use expression3

In [0]:
sql("select nvl2(null,'present','not_present')").show()

sql("select *,nvl2(age,'age provoided','get the age from user') as age_status from data_view").show()

## How to add a column in SQL

In [0]:
schema_struct="custid int,fname string,lname string,age int,prof string,erro_rec string"   
   
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/custsmodified.csv",header=True,schema=schema_struct,columnNameOfCorruptRecord="erro_rec",mode="dropmalformed")
cust_df.show(5)
cust_df.createOrReplaceTempView("View_cust")
#add a column in DSL - withColumn / select
cust_df.createOrReplaceTempView("View_cust")

view_name="View_cust"
spark.sql(f"select * from {view_name}").show(5)
spark.sql(f"select custid,concat(fname,' ',lname)as fullname,age, prof as Job_title from {view_name}").show(5)


In [0]:
cust_enriched_df=spark.sql(f"select custid,concat(fname,' ',lname)as fullname,age, profession as Job_title,current_date()as dt, 'anu' as created_by from {view_name}")
cust_enriched_df.show()

In [0]:
query=f"select custid,concat(fname,' ',lname)as fullname,age, profession as Job_title,current_date()as dt, 'anu' as created_by from {view_name}"
spark.sql(query).show()

In [0]:
spark.sql("select custid,upper(fname)as Upper_Fname,lower(lname)as Lower_Lname,initcap(concat(fname,' ',lname)) as Fullname from View_cust").show()


In [0]:
qry="select custid,substr(fname,1,3) as Fname_subStr from View_cust"
spark.sql(qry).show()

## DeDuplication

In [0]:
spark.sql("select * from View_cust").show(5)

spark.sql("select custid,count(*) from View_cust group by custid having count(*)>1").show()

spark.sql("select * from View_cust where custid in (4000003,4000001)").show()

spark.sql("select * from View_cust where custid is null or custid in (4000003,4000001)").show()


### Distinct - used to remove if the records have all null values

In [0]:
# remove duplicate 
# record level using distinct 
spark.sql("select * from View_cust").count()    #10004
spark.sql("select distinct * from View_cust").count()   #9998

spark.sql("select count(1) from View_cust").show()   # 10004

spark.sql(f"select distinct * from View_cust").filter("custid=4000001").show()

df2=cust_df.distinct() # unique records 
cust_df.filter("custid=4000001").show()
df2.filter("custid=4000001").show()

### DropDuplicate - column level duplicate 

In [0]:

dedep_col_df=cust_df.dropDuplicates(["custid"])
dedep_col_df.filter("custid in (4000003,4000001)").show()





## sub query -rownumber() / inline query/ qualify

In [0]:
%sql
-- sql based column level deduplication 
-- if custid repeat take the latest custid (based on age desc)


select *,row_number() over(partition by custid order by age desc) as rn from View_cust;

-- for which id it should take, which person's age is more then take that one
select * from(
select *,row_number() over(partition by custid order by age desc) as rn from View_cust limit 10
) as tbl where rn=1;


-- for which id it should take, which person's age is less then take that one
select * from(
select *,row_number() over(partition by custid order by age) as rn from View_cust limit 10
) as tbl where rn=1

## Case when

In [0]:
%sql
-- case when
-- age category -->age_cat if age>60 -->senior
-- if age >=40 --> middle
-- age<40 --> young

select * from View_cust limit 10;
select *, case when age>60 then 'senior' when age>=40 then 'middle' else 'young' end as age_cat from View_cust limit 10;

-- nvl logic using case when
select * ,case when age is null then 0 else age end as n_age from View_cust limit 10;


select *, case when age > 60 then 'senior_citizen'
when age >55 and prof='Lawyer' then 'spl category'
when age > 40 then 'middle_age'
else 'young_age'
end as age_cat from View_cust
where age > 55 and age <60;


select * from (
    select *, case when age > 60 then 'senior_citizen'
when age >55 and prof='Lawyer' then 'spl category'
when age > 40 then 'middle_age'
else 'young_age'
end as age_cat from View_cust
where age > 55 and age <60)where age_cat='spl category'



In [0]:
query="""select *, case when age > 60 then 'senior_citizen' when age >55 and prof='Lawyer' then 'spl category' when age > 40 then 'middle_age' else 'young_age' end as age_cat from View_cust where age > 55 and age <60""";

spark.sql(query).show(10)
spark.sql(query).filter("age_cat='spl category'").show()



##  UDF

In [0]:
%sql
create or replace temporary view cust_view_dropMal
(
    cust_id int,
    fname string,
    lname string,
    age int,
    Prof string,
    _errors string
)
using csv
options (
    header="true",
    inferSchema="True",
    path="/Volumes/izwd37dev/wd37db/rawdatta/csv/custsmodified.csv",
    mode="dropMalformed",
    columnNameOfCorruptRecord="_errors"
);
select * from cust_view_dropMal limit 10;

In [0]:
#udf 

def convert_upper(value):
    if value is None:
        return None  
    else:
        return value.upper()



# test the function
print(convert_upper("hello"))
print(convert_upper(None))



In [0]:
#drop the error columns usoing .drop function
from pyspark.sql.types import StringType
cust_df=spark.sql("select * from cust_view_dropMal limit 10").drop("_errors")
cust_df.show(5)

#sql udf
#Syntax: sql.udf.register(sql_udf_func_name,python_func,return_type)
#step 1: create python function
#step 2: register the funciton with spark.udf.register
#step 3: use the function with registered name

spark.udf.register("sql_udf_upper",convert_upper)
df2=spark.sql("select cust_id,fname,lname,age,Prof,sql_udf_upper(fname) as fname_upper,sql_udf_upper(nvl(lname,null)) as lname_upper from cust_view_dropMal limit 10")
df2.show()

###  lambda function in UDF

In [0]:
#using udf get the age and return with age*2
# default udf return type is string
#whatever the udf function we create, it will work only for this particular session, so other notebook if we are using, it will not work



spark.udf.register("double_age",lambda age:age*2)
df=spark.sql("select cust_id,fname,lname,age,Prof,double_age(nvl(age,0))as double_age from cust_view_dropMal limit 10")
df.show(10)
df.printSchema()

In [0]:
spark.range(10).createOrReplaceTempView("View_range")
spark.sql("select *,double_age(id)as id2 from View_range").show()

In [0]:
# in DSL (withColumn,withColumnRenamed,drop,select,selectExpr)
# for adding a column - withColumn,  select,
# rename the column - withColumnRenamed, 
# #remove the column - drop,
# using sql expression in DSL - selectExpr

#in SQL - all with (select)
spark.sql("select cust_id,fname,lname,age,Prof,double_age(nvl(age,0))as double_age,'testuser'as user from cust_view_dropMal limit 10").show()



## TypeCasting

In [0]:
#typecasting changing the double_age into integer
df=spark.sql("select cust_id,fname,lname,age,Prof,cast(double_age(nvl(age,0))as integer)as double_age from cust_view_dropMal")
df.show(5)
df.printSchema()



## Joins

### Left Join

In [0]:


emp_data=[(100,"A0",25),(101,"A1",36),(102,"A2",26),(103,"A3",32),(104,"A4",12),(105,"A5",54),(106,"A6",32)]

city_data=[(100,"chn"),(104,"blr"),(105,"hyd"),(106,"mum"),(107,"hyd"),(110,"del")]

emp_data=spark.createDataFrame(emp_data,['eid','ename','age'])
city_data=spark.createDataFrame(city_data,['eid','city'])
emp_data.createOrReplaceTempView("emp")
city_data.createOrReplaceTempView("city")

spark.sql("select * from emp").show()
spark.sql("select * from city").show()

# Join_syntax:
#select columns from left_table join|left|right|anti|semi|full|outer|cross|equi right_table on join condition


#need all emp with matching city, if emp doesn't have matching city mark it as NA
#left join syntax:
#select columns from left_table left join right_table on join condition

query="select * from emp e left join city c on e.eid=c.eid"  #shows two eid columns without NA for null city
#or 
query="select e.*,coalesce(c.city,'NA')as city from emp e left join city c on e.eid=c.eid"
#or 
query="select e.eid,e.ename,e.age,nvl(c.city,'NA')as city from emp e left join city c on e.eid=c.eid"

spark.sql(query).show()



### right join

In [0]:
#need all city data from city table, if city doesn't have matching emp mark it as NA
#right join syntax:
#select columns from left_table right join right_table on join condition

query="select * from emp e right join city c on e.eid=c.eid"  

query="select c.eid,coalesce(e.ename,'NA')as ename,coalesce(e.age,0)as age,c.city from emp e right join city c on e.eid=c.eid"

spark.sql(query).show()






### Inner join / Equi join

In [0]:

#Inner join - only common records from both sides
#keyword is join / inner join

query="select * from emp e join city c on e.eid=c.eid"  

query="select c.eid,coalesce(e.ename,'NA')as ename,coalesce(e.age,0)as age,c.city from emp e inner join city c on e.eid=c.eid"

spark.sql(query).show()

### Full join

In [0]:
#full join - comnining all records from both the table

query="select * from emp e full join city c on e.eid=c.eid"  

query="select c.eid,coalesce(e.ename,'NA')as ename,coalesce(e.age,0)as age,c.city from emp e full join city c on e.eid=c.eid"

spark.sql(query).show()

### left semi

In [0]:
#left semi and left anti - both these joins returning the records with left table alone

#left semi join - left side table alone it will show by comparing the matching records of right table

#need only emp details who have city values 
query="select * from emp e left semi join city c on e.eid=c.eid"  

spark.sql(query).show()

### left anti

In [0]:
#left anti join - shows left side table alone by comparing the non matching records of right table

#need only emp details who have not provided city values 
query="select * from emp e left anti join city c on e.eid=c.eid"  

spark.sql(query).show()

### self join

In [0]:
# self join - joining the same table


data = [
    (101,"John",104,"IT"),
    (102,"Alice",104,"IT"),
    (103,"Bob",105,"HR"),
    (104,"David",106,"IT"),
    (105,"Emma",106,"HR"),
    (106,"James",None,"Management")
]

cols=["eid","ename","manager_id","dept"]

emp_df=spark.createDataFrame(data,cols)

emp_df.createOrReplaceTempView("emp_table")

spark.sql("select * from emp_table").show()
#get the eid,ename,manager name,dept from the table
#self join - joining the same table

query="""
select * from emp_table em  join emp_table mgr on em.manager_id=mgr.eid
"""

query="""
select em.eid,em.ename,mgr.ename as manager_name,em.dept from emp_table em  join emp_table mgr on em.manager_id=mgr.eid
"""



spark.sql(query).show()

In [0]:
%sql
-- implementing the same self join using sql syntax
select e.eid,e.ename,mgr.ename,e.dept from emp_table e join emp_table mgr on e.manager_id = mgr.eid;

select e.eid,e.ename,mgr.ename,e.dept from emp_table e left join emp_table mgr on e.manager_id = mgr.eid;

select e.eid,e.ename,nvl(mgr.ename,'CEO') as manager,e.dept from emp_table e left join emp_table mgr on e.manager_id = mgr.eid